# Generate Fine-Tuning Dataset

This notebook creates a chat fine-tuning dataset with an 80/20 mix:

- 80% template-generated Q&A from product data.
- 20% Groq-generated Q&A for more natural phrasing.

API generation is disabled by default. Set `RUN_API = True` to call Groq.


In [ ]:
import json
import os
import re
import time
from pathlib import Path
import os
import pandas as pd

PROJECT_ROOT = Path("/Users/huybui/Desktop/Smart Shopping Chatbot")
PRODUCTS_PATH = PROJECT_ROOT / "processed" / "catalog.csv"
OUTPUT_PATH = PROJECT_ROOT / "processed" / "fine_tuning_dataset.jsonl"

TARGET_QA_COUNT = 3000
TEMPLATE_RATIO = 0.8
API_RATIO = 0.2
QA_PER_PRODUCT = 2

TEMPLATE_QA_COUNT = int(TARGET_QA_COUNT * TEMPLATE_RATIO)
API_QA_COUNT = TARGET_QA_COUNT - TEMPLATE_QA_COUNT
TEMPLATE_MAX_PRODUCTS = TEMPLATE_QA_COUNT // QA_PER_PRODUCT
API_MAX_PRODUCTS = API_QA_COUNT // QA_PER_PRODUCT
API_START_INDEX = TEMPLATE_MAX_PRODUCTS

RUN_API = True
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.1-8b-instant")
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
SLEEP_SECONDS = 1.0
products = pd.read_csv(PRODUCTS_PATH).fillna("")
print("Total products:", len(products))
print("Template target Q&A:", TEMPLATE_QA_COUNT)
print("API target Q&A:", API_QA_COUNT)
products.head()


## Helpers


In [ ]:
CATEGORY_LABELS = {
    "dien_thoai": "điện thoại",
    "laptop": "laptop",
    "phu_kien": "phụ kiện",
    "tai_nghe_bluetooth": "tai nghe bluetooth",
    "man_hinh": "màn hình",
    "may_tinh_bang": "máy tính bảng",
    "loa": "loa",
    "pc": "PC",
    "micro": "micro",
}


def compact_text(value: object, max_chars: int = 900) -> str:
    text = str(value or "")
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(" ", 1)[0].strip() + "..."


def category_label(category: str) -> str:
    return CATEGORY_LABELS.get(str(category), str(category).replace("_", " "))


def price_sentence(price: str) -> str:
    price = str(price or "").strip()
    if not price or price.lower() in {"liên hệ", "lien he", "đang cập nhật", "dang cap nhat"}:
        return "Giá hiện tại cần liên hệ cửa hàng để được cập nhật chính xác."
    return f"Giá tham khảo hiện tại là {price}."


def brand_sentence(brand: str) -> str:
    brand = str(brand or "").strip()
    return f"Sản phẩm thuộc thương hiệu {brand}." if brand else "Dữ liệu hiện chưa có thông tin thương hiệu rõ ràng."


def specs_sentence(specs_text: str) -> str:
    specs = compact_text(specs_text, max_chars=1000)
    return f"Thông số chính gồm: {specs}." if specs else "Dữ liệu hiện chưa có thông số kỹ thuật chi tiết."


def product_summary(row: pd.Series) -> str:
    name = row["ten_san_pham"]
    category = category_label(row["category"])
    description = compact_text(row.get("mo_ta_ngan", ""), max_chars=450)
    parts = [
        f"{name} là sản phẩm thuộc danh mục {category}.",
        brand_sentence(row.get("thuong_hieu", "")),
        price_sentence(row.get("gia_ban", "")),
    ]
    if description:
        parts.append(f"Mô tả ngắn: {description}.")
    parts.append(specs_sentence(row.get("specs_text", "")))
    if row.get("url", ""):
        parts.append(f"Link sản phẩm: {row['url']}")
    return " ".join(parts)


def product_context(row: pd.Series) -> str:
    return f"""
product_id: {row.get('product_id', '')}
Tên sản phẩm: {row.get('ten_san_pham', '')}
Thương hiệu: {row.get('thuong_hieu', '')}
Danh mục: {category_label(row.get('category', ''))}
Giá bán: {row.get('gia_ban', '') or 'Chưa có giá'}
Mô tả ngắn: {compact_text(row.get('mo_ta_ngan', ''), max_chars=350)}
Thông số kỹ thuật: {compact_text(row.get('specs_text', ''), max_chars=900)}
URL: {row.get('url', '')}
""".strip()


## Template Q&A


In [ ]:
def qa_templates(row: pd.Series) -> list[dict]:
    product_id = row["product_id"]
    name = row["ten_san_pham"]
    category = category_label(row["category"])
    brand = str(row.get("thuong_hieu", "")).strip()
    price = price_sentence(row.get("gia_ban", ""))
    specs = specs_sentence(row.get("specs_text", ""))
    summary = product_summary(row)
    brand_part = f" của thương hiệu {brand}" if brand else ""

    return [
        {
            "product_id": product_id,
            "question": f"{name} có thông tin gì nổi bật?",
            "answer": summary,
            "source": "template_generated",
        },
        {
            "product_id": product_id,
            "question": f"Thông số kỹ thuật của {name} là gì?",
            "answer": f"{name} là {category}{brand_part}. {specs} {price}",
            "source": "template_generated",
        },
        {
            "product_id": product_id,
            "question": f"Giá {name} bao nhiêu và xem ở đâu?",
            "answer": f"{price} Bạn có thể xem chi tiết sản phẩm tại: {row.get('url', '')}",
            "source": "template_generated",
        },
    ]


def generate_template_items(products_df: pd.DataFrame) -> list[dict]:
    rows = []
    selected = products_df.head(TEMPLATE_MAX_PRODUCTS).copy()
    for _, row in selected.iterrows():
        rows.extend(qa_templates(row)[:QA_PER_PRODUCT])
    return rows


template_items = generate_template_items(products)
print("Template Q&A:", len(template_items))
template_items[:2]


## Groq API Q&A


In [ ]:
SYSTEM_PROMPT = """
Bạn là trợ lý tạo dữ liệu huấn luyện cho chatbot tư vấn mua sắm.
Chỉ dùng thông tin sản phẩm được cung cấp. Không bịa giá, thông số, tồn kho, khuyến mãi hoặc đánh giá.
Nếu giá là 'Liên hệ' hoặc thiếu giá, hãy nói rằng giá cần liên hệ cửa hàng.
Trả về JSON array hợp lệ, không markdown, không giải thích thêm.
""".strip()


def build_api_prompt(row: pd.Series, qa_count: int = 2) -> str:
    return f"""
Hãy tạo {qa_count} cặp câu hỏi - câu trả lời tiếng Việt để fine-tune chatbot bán hàng.

Yêu cầu:
- Câu hỏi phải tự nhiên như khách hàng thật.
- Câu trả lời ngắn gọn, chính xác, chỉ dựa trên dữ liệu sản phẩm.
- Nên có kiểu hỏi tư vấn nhu cầu, hỏi thông số, hoặc hỏi giá/link mua.
- Mỗi object có đúng 3 field: product_id, question, answer.
- Trả về JSON array.

Dữ liệu sản phẩm:
{product_context(row)}
""".strip()


def parse_json_array(text: str) -> list[dict]:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    start = text.find("[")
    end = text.rfind("]")
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"No JSON array found: {text[:300]}")
    data = json.loads(text[start : end + 1])
    if not isinstance(data, list):
        raise ValueError("Expected JSON array")
    return data


def get_groq_client():
    from openai import OpenAI

    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("Missing GROQ_API_KEY. Set it before running API cells.")
    return OpenAI(api_key=api_key, base_url=GROQ_BASE_URL)


def generate_api_qa_for_product(client, row: pd.Series, qa_count: int = 2) -> list[dict]:
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_api_prompt(row, qa_count=qa_count)},
        ],
        temperature=0.4,
    )

    content = response.choices[0].message.content
    items = parse_json_array(content)
    cleaned = []
    for item in items:
        product_id = str(item.get("product_id") or row.get("product_id", "")).strip()
        question = str(item.get("question", "")).strip()
        answer = str(item.get("answer", "")).strip()
        if product_id and question and answer:
            cleaned.append({
                "product_id": product_id,
                "question": question,
                "answer": answer,
                "source": "groq_generated",
            })
    return cleaned


In [ ]:
def generate_api_items(products_df: pd.DataFrame) -> list[dict]:
    client = get_groq_client()
    all_items = []
    selected = products_df.iloc[API_START_INDEX : API_START_INDEX + API_MAX_PRODUCTS].copy()

    for offset, (_, row) in enumerate(selected.iterrows(), start=1):
        try:
            items = generate_api_qa_for_product(client, row, qa_count=QA_PER_PRODUCT)
            all_items.extend(items[:QA_PER_PRODUCT])
            print(f"{offset}/{len(selected)} {row['product_id']}: {len(items)} Q&A")
        except Exception as error:
            print(f"Failed {row.get('product_id', offset)}: {error}")
        time.sleep(SLEEP_SECONDS)

    return all_items


if RUN_API:
    api_items = generate_api_items(products)
else:
    api_items = []
    print("RUN_API is False. API Q&A not generated.")

print("API Q&A:", len(api_items))


## Convert to Fine-Tuning JSONL


In [ ]:
FINETUNE_SYSTEM_MESSAGE = """
Bạn là chatbot tư vấn mua sắm. Trả lời ngắn gọn, chính xác, chỉ dựa trên dữ liệu sản phẩm đã biết. Không bịa giá, tồn kho, khuyến mãi hoặc thông số.
""".strip()


def to_finetune_record(item: dict) -> dict:
    return {
        "messages": [
            {"role": "system", "content": FINETUNE_SYSTEM_MESSAGE},
            {"role": "user", "content": item["question"]},
            {"role": "assistant", "content": item["answer"]},
        ],
        "metadata": {
            "product_id": item["product_id"],
            "source": item.get("source", "unknown"),
        },
    }


qa_items = template_items + api_items
fine_tune_records = [to_finetune_record(item) for item in qa_items]
print("Total Q&A:", len(fine_tune_records))
fine_tune_records[:2]


## Save JSONL


In [ ]:
def save_jsonl(records: list[dict], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")


save_jsonl(fine_tune_records, OUTPUT_PATH)
print("Saved:", OUTPUT_PATH)
print("Records:", len(fine_tune_records))
